# Phase 3 + 4 — 3D Access Graph & Analysis (Intact Section)
**Marina Osmolovska — MaCAD S3 Graph ML Final**

Soviet-era 87-02 panel block-section, Kyiv. Intact section only.

**Phase 3:** Builds adjacency graph and access graph (via door apertures).  
**Phase 4:** Centrality metrics + depth-from-entrance table — drives slab heights in Phase 5.

## 1. Import the needed classes

In [ ]:
from topologicpy.Vertex import Vertex
from topologicpy.Edge import Edge
from topologicpy.Face import Face
from topologicpy.Cell import Cell
from topologicpy.CellComplex import CellComplex
from topologicpy.Topology import Topology
from topologicpy.Dictionary import Dictionary
from topologicpy.Graph import Graph
from topologicpy.Helper import Helper
import os

## 2. Check the TopologicPy version

In [12]:
print("This notebook requires topologicpy version 0.9.18 or newer.")
print(Helper.Version())

This notebook requires topologicpy version 0.9.18 or newer.
The version that you are using (0.9.43) is OLDER than the latest version (0.9.50) from PyPI. Please consider upgrading to the latest version.


## 3. Set your renderer
* Visual Studio Code: `"vscode"`
* Google Colab: `"colab"`
* Browser: `"browser"`

In [13]:
renderer = "vscode"

## 4. Set paths to OBJ files

In [14]:
BASE       = os.path.dirname(os.path.abspath("__file__"))  # Final/ directory
OBJ_DIR    = os.path.join(BASE, "assets", "3d")
ROOMS_OBJ  = os.path.join(OBJ_DIR, "rooms.obj")
DOORS_ENT  = os.path.join(OBJ_DIR, "doors_entrance.obj")
DOORS_DOOR = os.path.join(OBJ_DIR, "doors_door.obj")
DOORS_PASS = os.path.join(OBJ_DIR, "doors_passage.obj")

print("OBJ_DIR:", OBJ_DIR)
for p in [ROOMS_OBJ, DOORS_ENT, DOORS_DOOR, DOORS_PASS]:
    print(os.path.basename(p), "exists:", os.path.exists(p))

OBJ_DIR: d:\Marina\MaCAD 2025\3 SEMESTER\Graph ML\graph_ml\graph_ml\Final\assets\3d
rooms.obj exists: True
doors_entrance.obj exists: True
doors_door.obj exists: True
doors_passage.obj exists: True


## 5. Load rooms OBJ

In [15]:
objects = Topology.ByOBJPath(ROOMS_OBJ)
print("Objects is a list")
print(objects)

Objects is a list
[<topologic_core.Cluster object at 0x000001F5CC654D70>, <topologic_core.Cluster object at 0x000001F5CCCD6C70>, <topologic_core.Cluster object at 0x000001F5CCCD7330>, <topologic_core.Cluster object at 0x000001F5CCCCDB70>, <topologic_core.Cluster object at 0x000001F5CC6407B0>, <topologic_core.Cluster object at 0x000001F5CCCCD8B0>, <topologic_core.Cluster object at 0x000001F5CCBEAC30>, <topologic_core.Cluster object at 0x000001F5CCBE9270>, <topologic_core.Cluster object at 0x000001F5CCBEAB70>, <topologic_core.Cluster object at 0x000001F5CCC01330>, <topologic_core.Cluster object at 0x000001F5CCBE9070>, <topologic_core.Cluster object at 0x000001F5CCBEB0B0>, <topologic_core.Cluster object at 0x000001F5CCBE9330>, <topologic_core.Cluster object at 0x000001F5CCBEB1F0>, <topologic_core.Cluster object at 0x000001F5CCBEB870>, <topologic_core.Cluster object at 0x000001F5CCBE9170>, <topologic_core.Cluster object at 0x000001F5CCBE81B0>, <topologic_core.Cluster object at 0x000001F5CC

In [16]:
# Diagnostic — run this before cell 6 to see what was loaded
print("--- OBJ object diagnostic ---")
for i, obj in enumerate(objects):
    d     = Topology.Dictionary(obj)
    name  = Dictionary.ValueAtKey(d, "name")  or ""
    group = Dictionary.ValueAtKey(d, "group") or ""
    faces = Topology.Faces(obj)
    ttype = type(obj).__name__
    print(f"[{i}]  type={ttype:12s}  faces={len(faces):3d}  name='{name}'  group='{group}'")


--- OBJ object diagnostic ---
[0]  type=Cluster       faces=  0  name='default'  group='default'
[1]  type=Cluster       faces=  6  name='livingroom_9_33'  group='livingroom_9_33'
[2]  type=Cluster       faces=  6  name='corridor_9_32'  group='corridor_9_32'
[3]  type=Cluster       faces=  8  name='corridor_9_31'  group='corridor_9_31'
[4]  type=Cluster       faces=  6  name='bathroom_9_30'  group='bathroom_9_30'
[5]  type=Cluster       faces=  6  name='bedroom_9_29'  group='bedroom_9_29'
[6]  type=Cluster       faces=  6  name='livingroom_9_28'  group='livingroom_9_28'
[7]  type=Cluster       faces=  8  name='kitchen_9_27'  group='kitchen_9_27'
[8]  type=Cluster       faces=  6  name='storeroom_9_26'  group='storeroom_9_26'
[9]  type=Cluster       faces=  8  name='kitchen_9_25'  group='kitchen_9_25'
[10]  type=Cluster       faces=  6  name='bathroom_9_24'  group='bathroom_9_24'
[11]  type=Cluster       faces=  6  name='bathroom_9_23'  group='bathroom_9_23'
[12]  type=Cluster       fac

## 6. Prepare room dictionaries

Reads `room_type` from the OBJ object name (`o` line). Assigns a colour for visualisation and a numeric label for ML.

Fallback: if `name` is empty, tries the `group` key (layer-as-group OBJ export).

In [17]:
ROOM_LABEL = {
    "bedroom": 0, "livingroom": 1, "kitchen": 2,
    "corridor": 3, "stairs": 4, "storeroom": 5, "bathroom": 6, "balcony": 7
}
ROOM_COLOR = {
    "bedroom": "blue", "livingroom": "yellow", "kitchen": "orange",
    "corridor": "yellow", "stairs": "red",
    "storeroom": "purple", "bathroom": "purple", "balcony": "green"
}

def parse_room_type(s):
    """Extract room type from names like 'corridor_9_32' or plain 'corridor'."""
    s = s.strip().lower()
    if s in ROOM_LABEL:
        return s
    base = s.split("_")[0]
    return base if base in ROOM_LABEL else None

cells_list = []
selectors  = []

for obj in objects:
    d = Topology.Dictionary(obj)
    name  = Dictionary.ValueAtKey(d, "name")  or ""
    group = Dictionary.ValueAtKey(d, "group") or ""

    room_type = parse_room_type(name) or parse_room_type(group)
    if room_type is None:
        print(f"SKIP: name='{name}'  group='{group}' — neither recognised")
        continue

    faces = Topology.Faces(obj)
    cell_name = (name or group).strip()
    print(f"  {room_type} ({cell_name}): {len(faces)} faces")

    c = Cell.ByFaces(faces) if len(faces) > 1 else faces[0]
    if c is None:
        print(f"  ERROR: Cell.ByFaces returned None for '{cell_name}'")
        continue
    c = Topology.RemoveCollinearEdges(c)
    if c is None:
        print(f"  ERROR: RemoveCollinearEdges returned None for '{cell_name}'")
        continue

    color = ROOM_COLOR[room_type]
    label = ROOM_LABEL[room_type]
    d = Dictionary.SetValuesAtKeys(d,
        ["room_type", "cell_name", "label", "color", "vertex_size"],
        [room_type,   cell_name,   label,  color,  20])
    s = Topology.InternalVertex(c)
    s = Topology.SetDictionary(s, d)
    selectors.append(s)
    cells_list.append(c)
    print(f"  OK: {room_type}")

print(f"\n{len(cells_list)} valid cells loaded")

SKIP: name='default'  group='default' — neither recognised
  livingroom (livingroom_9_33): 6 faces
  OK: livingroom
  corridor (corridor_9_32): 6 faces
  OK: corridor
  corridor (corridor_9_31): 8 faces
  OK: corridor
  bathroom (bathroom_9_30): 6 faces
  OK: bathroom
  bedroom (bedroom_9_29): 6 faces
  OK: bedroom
  livingroom (livingroom_9_28): 6 faces
  OK: livingroom
  kitchen (kitchen_9_27): 8 faces
  OK: kitchen
  storeroom (storeroom_9_26): 6 faces
  OK: storeroom
  kitchen (kitchen_9_25): 8 faces
  OK: kitchen
  bathroom (bathroom_9_24): 6 faces
  OK: bathroom
  bathroom (bathroom_9_23): 6 faces
  OK: bathroom
  balcony (balcony_9_22): 6 faces
  OK: balcony
  balcony (balcony_9_21): 6 faces
  OK: balcony
  bedroom (bedroom_9_20): 6 faces
  OK: bedroom
  bathroom (bathroom_9_19): 6 faces
  OK: bathroom
  bedroom (bedroom_9_18): 6 faces
  OK: bedroom
  livingroom (livingroom_9_17): 6 faces
  OK: livingroom
  corridor (corridor_9_16): 6 faces
  OK: corridor
  corridor (corridor_9_

## 7. Build CellComplex

In [18]:
house = CellComplex.ByCells(cells_list)
house = Topology.TransferDictionariesBySelectors(house, selectors, tranCells=True)
house_cells = Topology.Cells(house)
for house_cell in house_cells:
    d = Topology.Dictionary(house_cell)
    print(Dictionary.Keys(d), Dictionary.Values(d))

['cell_name', 'color', 'group', 'label', 'material', 'name', 'opacity', 'room_type', 'vertex_size'] ['livingroom_9_33', 'yellow', 'livingroom_9_33', 1, '', 'livingroom_9_33', 1.0, 'livingroom', 20]
['cell_name', 'color', 'group', 'label', 'material', 'name', 'opacity', 'room_type', 'vertex_size'] ['balcony_9_22', 'green', 'balcony_9_22', 7, '', 'balcony_9_22', 1.0, 'balcony', 20]
['cell_name', 'color', 'group', 'label', 'material', 'name', 'opacity', 'room_type', 'vertex_size'] ['bedroom_9_20', 'blue', 'bedroom_9_20', 0, '', 'bedroom_9_20', 1.0, 'bedroom', 20]
['cell_name', 'color', 'group', 'label', 'material', 'name', 'opacity', 'room_type', 'vertex_size'] ['livingroom_8_33', 'yellow', 'livingroom_8_33', 1, '', 'livingroom_8_33', 1.0, 'livingroom', 20]
['cell_name', 'color', 'group', 'label', 'material', 'name', 'opacity', 'room_type', 'vertex_size'] ['kitchen_9_25', 'orange', 'kitchen_9_25', 2, '', 'kitchen_9_25', 1.0, 'kitchen', 20]
['cell_name', 'color', 'group', 'label', 'materia

## 8. Show geometry

In [19]:
Topology.Show(house_cells, faceColorKey="color", faceOpacity=0.7, renderer=renderer)

## 9. Adjacency graph (g1)

Connects every pair of rooms that share a face — includes through-slab connections.
Shown here as a comparison baseline; the access graph (g2) is what drives the design.

In [20]:
g1 = Graph.ByTopology(house)
verts = Graph.Vertices(g1)
for v in verts:
    d = Topology.Dictionary(v)
    print(Dictionary.Keys(d), Dictionary.Values(d))

['category', 'cell_name', 'color', 'group', 'index', 'label', 'material', 'name', 'ontology_class', 'ontology_uri', 'opacity', 'room_type', 'vertex_size'] ['node', 'livingroom_9_33', 'yellow', 'livingroom_9_33', 0, 1, '', 'livingroom_9_33', 'top:Node', 'http://w3id.org/topologicpy#Node', 1.0, 'livingroom', 20]
['category', 'cell_name', 'color', 'group', 'index', 'label', 'material', 'name', 'ontology_class', 'ontology_uri', 'opacity', 'room_type', 'vertex_size'] ['node', 'balcony_9_22', 'green', 'balcony_9_22', 1, 7, '', 'balcony_9_22', 'top:Node', 'http://w3id.org/topologicpy#Node', 1.0, 'balcony', 20]
['category', 'cell_name', 'color', 'group', 'index', 'label', 'material', 'name', 'ontology_class', 'ontology_uri', 'opacity', 'room_type', 'vertex_size'] ['node', 'bedroom_9_20', 'blue', 'bedroom_9_20', 2, 0, '', 'bedroom_9_20', 'top:Node', 'http://w3id.org/topologicpy#Node', 1.0, 'bedroom', 20]
['category', 'cell_name', 'color', 'group', 'index', 'label', 'material', 'name', 'ontology

## 10. Show geometry and adjacency graph

In [29]:
Topology.Show(g1, house, vertexSizeKey="vertex_size", vertexColorKey="color", backgroundColor="black", renderer=renderer)

## 11. Load door apertures

Three separate OBJ files — one per door type. Each face gets a `door_type` dictionary key
that S06-15B reads later to build edge features.

| file | door_type | meaning |
|---|---|---|
| doors_entrance.obj | `entrance_door` | apartment entrance from landing |
| doors_door.obj | `door` | room-to-room leaf door |
| doors_passage.obj | `passage` | open threshold, stair landing to corridor |

In [23]:
apertures = []

def load_apertures(obj_path, door_type, color):
    objs = Topology.ByOBJPath(obj_path)
    count = 0
    for obj in objs:
        faces = Topology.Faces(obj)
        if not faces:
            continue
        for face in faces:
            face = Topology.RemoveCollinearEdges(face)
            d = Dictionary.ByKeysValues(
                ["door_type", "color", "vertex_size"],
                [door_type,   color,   15])
            face = Topology.SetDictionary(face, d)
            apertures.append(face)
            count += 1
    print(f"  {door_type}: {count} apertures")

load_apertures(DOORS_ENT,  "entrance_door", "green")
load_apertures(DOORS_DOOR, "door",          "brown")
load_apertures(DOORS_PASS, "passage",       "grey")

print(f"\n{len(apertures)} apertures total")

  entrance_door: 36 apertures
  door: 252 apertures
  passage: 18 apertures

306 apertures total


## 12. Add apertures to the CellComplex

In [24]:
house = Topology.AddApertures(house, apertures, subTopologyType="face")

## 13. Access graph (g2) — via shared apertures only

`direct=False` — do NOT connect every shared face (that would give adjacency again).
`viaSharedApertures=True` — connect ONLY cells sharing a door aperture.
`toExteriorApertures=False` — ignore exterior faces (no windows in the access graph).

This single switch — `direct=False` + `viaSharedApertures=True` — is the whole difference
between the adjacency graph above and the access graph.

In [26]:
g2 = Graph.ByTopology(
    house,
    direct=False,
    viaSharedApertures=True,
    toExteriorApertures=False,
    useInternalVertex=True,
    tolerance=0.0001
)
verts = Graph.Vertices(g2)
for v in verts:
    d = Topology.Dictionary(v)
    print(Dictionary.Keys(d), Dictionary.Values(d))

['category', 'cell_name', 'color', 'group', 'index', 'label', 'material', 'name', 'ontology_class', 'ontology_uri', 'opacity', 'room_type', 'vertex_size'] ['node', 'livingroom_9_33', 'yellow', 'livingroom_9_33', 0, 1, '', 'livingroom_9_33', 'top:Node', 'http://w3id.org/topologicpy#Node', 1.0, 'livingroom', 20]
['category', 'cell_name', 'color', 'group', 'index', 'label', 'material', 'name', 'ontology_class', 'ontology_uri', 'opacity', 'room_type', 'vertex_size'] ['node', 'balcony_9_22', 'green', 'balcony_9_22', 1, 7, '', 'balcony_9_22', 'top:Node', 'http://w3id.org/topologicpy#Node', 1.0, 'balcony', 20]
['category', 'cell_name', 'color', 'group', 'index', 'label', 'material', 'name', 'ontology_class', 'ontology_uri', 'opacity', 'room_type', 'vertex_size'] ['node', 'bedroom_9_20', 'blue', 'bedroom_9_20', 2, 0, '', 'bedroom_9_20', 'top:Node', 'http://w3id.org/topologicpy#Node', 1.0, 'bedroom', 20]
['category', 'cell_name', 'color', 'group', 'index', 'label', 'material', 'name', 'ontology

## 14. Show access graph

In [28]:
Topology.Show(house, apertures, g2, vertexSizeKey="vertex_size", vertexColorKey="color", backgroundColor="black", renderer=renderer)

---
## Phase 4 — Graph Analysis

Runs directly on `g2` already in memory. No reload needed.

| Metric | What it tells you |
|---|---|
| `depth_from_entrance` | hop count from staircase → drives slab Z height in Phase 5 |
| `betweenness_centrality` | rooms that act as gateways / circulation spine |
| `closeness_centrality` | rooms most accessible from everywhere |
| `degree_centrality` | rooms with most direct connections |

In [ ]:
## 15. Centrality metrics (stored in vertex dictionaries)
g2 = Graph.BetweennessCentrality(g2)
g2 = Graph.ClosenessCentrality(g2)
g2 = Graph.DegreeCentrality(g2)
print("Centralities computed.")

In [ ]:
## 16. Find entrance vertex (staircase/corridor side of entrance_door edge)
vertices   = Graph.Vertices(g2)
edges_list = Graph.Edges(g2)

entrance_vertex = None
for e in edges_list:
    de = Topology.Dictionary(e)
    if (Dictionary.ValueAtKey(de, "door_type") or "") == "entrance_door":
        for endpoint in [Edge.StartVertex(e), Edge.EndVertex(e)]:
            for gv in vertices:
                if Vertex.Distance(endpoint, gv) < 0.001:
                    dv = Topology.Dictionary(gv)
                    if (Dictionary.ValueAtKey(dv, "room_type") or "") in ("stairs", "corridor"):
                        entrance_vertex = gv
                        break
            if entrance_vertex:
                break
    if entrance_vertex:
        break

if entrance_vertex is None:
    for gv in vertices:
        dv = Topology.Dictionary(gv)
        if (Dictionary.ValueAtKey(dv, "room_type") or "") == "stairs":
            entrance_vertex = gv
            print("WARNING: no entrance_door edge found — using staircase node")
            break

dv = Topology.Dictionary(entrance_vertex)
print("Entrance node room_type:", Dictionary.ValueAtKey(dv, "room_type"))

In [ ]:
## 17. Depth-from-entrance table (copy the depth column → slab_Z = -k × depth in GH)
results = []
for v in vertices:
    dv    = Topology.Dictionary(v)
    rtype = Dictionary.ValueAtKey(dv, "room_type")               or "?"
    cname = Dictionary.ValueAtKey(dv, "cell_name")               or ""
    btw   = Dictionary.ValueAtKey(dv, "betweenness_centrality")  or 0
    clo   = Dictionary.ValueAtKey(dv, "closeness_centrality")    or 0
    deg   = Dictionary.ValueAtKey(dv, "degree_centrality")       or 0
    depth = Graph.TopologicalDistance(g2, entrance_vertex, v)
    results.append((rtype, cname, depth, btw, clo, deg))

results.sort(key=lambda x: (x[2], x[0]))

print(f"{'room_type':<14} {'cell_name':<22} {'depth':>5} {'betweenness':>12} {'closeness':>10} {'degree':>8}")
print("-" * 75)
for rtype, cname, depth, btw, clo, deg in results:
    print(f"{rtype:<14} {cname:<22} {depth:>5} {btw:>12.4f} {clo:>10.4f} {deg:>8.4f}")